# Module 4 - The Controlled Sandbox

**Time:** 15:30-17:00  
**Tool focus:** YSocial plus `ysights`

This notebook assumes a YSocial simulation database is available and keeps the `ysights` analysis explicit, so the follow network, mention network, and content signals can be tied back to enclaves and polarisation.


In [ ]:
!pip install ysights
#wget -O data.zip "https://data.d4science.net/mQAFv"
#!unzip data.zip

In [ ]:
from ysights import YDataHandler
import pandas as pd
import networkx as nx
from cdlib import algorithms, evaluation, viz, NodeClustering
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import Image, display
from collections import Counter

In [ ]:
DB_PATH = 'ysocial_exp.db'
handler = YDataHandler(str(DB_PATH))

In [ ]:
print(f'Simulation with {handler.number_of_agents()} agents')

## 1. Read the agents and attach their attributes


In [ ]:
agents = handler.agents().get_agents()

agents_df = pd.DataFrame(
    [
        {
            "agent_id": agent.id,
            "username": agent.username,
            "leaning": agent.leaning,
            "content_recsys": agent.recsys["content"],
            "social_recsys": agent.recsys["social"],
            "education": agent.education,
        }
        for agent in agents
    ]
)
agents_df.head()


## 2. Recover the follow and mention networks with `ysights`


In [ ]:
social_graph = handler.social_network()
mention_graph = handler.mention_network()

leaning_lookup = dict(zip(agents_df["agent_id"], agents_df["leaning"]))
nx.set_node_attributes(social_graph, leaning_lookup, "leaning")
nx.set_node_attributes(mention_graph, leaning_lookup, "leaning")

pd.DataFrame(
    [
        {"network": "follow", "nodes": social_graph.number_of_nodes(), "edges": social_graph.number_of_edges()},
        {"network": "mention", "nodes": mention_graph.number_of_nodes(), "edges": mention_graph.number_of_edges()},
    ]
)


## 3. Detect enclave-like clusters inside the simulation output


In [ ]:
social_graph_u = social_graph.to_undirected()
mention_graph_u = mention_graph.to_undirected()

follow_partition = algorithms.louvain(social_graph_u)
mention_partition = algorithms.louvain(mention_graph_u)
follow_partition.method_name = "follow_louvain"
mention_partition.method_name = "mention_louvain"


In [ ]:
partition_rows = []
for graph_name, graph_obj, partition in [
    ("follow-louvain", social_graph_u, follow_partition),
    ("mention-louvain", mention_graph_u, mention_partition),
]:
    partition_rows.append(
        {
            "partition": graph_name,
            "communities": len(partition.communities),
            "modularity": round(evaluation.newman_girvan_modularity(graph_obj, partition).score, 4),
            "internal_edge_density": round(evaluation.internal_edge_density(graph_obj, partition).score, 4),
            "conductance": round(evaluation.conductance(graph_obj, partition).score, 4),
            "hub_dominance": round(evaluation.hub_dominance(graph_obj, partition).score, 4),
        }
    )

pd.DataFrame(partition_rows)


In [ ]:
viz.plot_community_graph(social_graph_u, follow_partition, figsize=(8, 8), plot_labels=True)
plt.title('Follow-graph communities')
plt.show()

In [ ]:
viz.plot_community_graph(mention_graph_u, mention_partition, figsize=(8, 8), plot_labels=True)
plt.title('Mention-graph communities')
plt.show()

In [ ]:

def get_community_metrics(G, partition):
    def purity(nodes, G):
        labels = [G.nodes[n]["leaning"] for n in nodes if "leaning" in G.nodes[n]]
        return Counter(labels).most_common(1)[0][1] / len(labels) if labels else 0

    rows = []

    for i, community in enumerate(partition.communities):
        community = [n for n in community if n in G]
        if not community:
            continue

        rows.append({
            "algorithm": partition.method_name,
            "community_id": f"{partition.method_name}_{i}",
            "conductance": evaluation.conductance(
                G,
                NodeClustering([community], G, partition.method_name)
            ).score,
            "purity": purity(community, G),
            "size": len(community)
        })

    return pd.DataFrame(rows)


metrics_df = get_community_metrics(social_graph_u, follow_partition)

In [ ]:

plt.figure(figsize=(10, 8))
sns.kdeplot(x=metrics_df['conductance'], y=metrics_df['purity'],
            fill=True, cmap='inferno',
            levels=100, thresh=0.0001,

            )
plt.xlabel('Conductance')
plt.ylabel('Label Purity')
plt.title('Conductance vs. Label Purity')
plt.grid(None)

kde_plot_path = "module2_conductance_purity_kde.png"
plt.savefig(kde_plot_path, bbox_inches="tight")
display(Image(filename=str(kde_plot_path)))
plt.close('all')

In [ ]:
metrics_df = get_community_metrics(mention_graph_u, mention_partition)

plt.figure(figsize=(10, 8))
sns.kdeplot(x=metrics_df['conductance'], y=metrics_df['purity'],
            fill=True, cmap='inferno',
            levels=100, thresh=0.0001,

            )
plt.xlabel('Conductance')
plt.ylabel('Label Purity')
plt.title('Conductance vs. Label Purity')
plt.grid(None)


## 4. Inspect content signals with `ysights`


In [ ]:
hashtag_rows = []
for agent_id in agents_df["agent_id"]:
    for hashtag, count in handler.agent_hashtags(agent_id).items():
        hashtag_rows.append({"agent_id": agent_id, "hashtag": hashtag, "count": count})

hashtag_frame = pd.DataFrame(hashtag_rows).merge(
    agents_df[["agent_id", "leaning"]],
    on="agent_id",
    how="left",
)

hashtag_frame.groupby(["leaning", "hashtag"])["count"].sum().reset_index().sort_values(["leaning", "count"], ascending=[True, False])


In [ ]:
emotion_rows = []

for agent_id in agents_df["agent_id"]:
    for emotion, count in handler.agent_emotions(agent_id).items():
        emotion_rows.append({
            "agent_id": agent_id,
            "emotion": emotion,
            "count": count
        })

emotions_df = pd.DataFrame(emotion_rows)

In [ ]:
emotion_totals = (
    emotions_df
    .groupby("emotion")["count"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

plt.figure(figsize=(8, 4))
sns.barplot(data=emotion_totals, x="count", y="emotion")
plt.title("Total Emotion Counts")
plt.xlabel("Count")
plt.ylabel("Emotion")
plt.tight_layout()
plt.show()